In [1]:
import pandas as pd

# Define file paths for CVRP
EA_path = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\4_CVRPTW_dual_bounds_and_models\Solve_CVRPTW_time_limit\v1\CVRPTW_EA_dual_bound_verification_results_1800s_lim.csv"
Single_path = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\4_CVRPTW_dual_bounds_and_models\Solve_CVRPTW_time_limit\CVRPTW_single_dual_bound_selected_results_1800s_lim.csv"
output_filename = 'aggregated_cvrptw_results.csv'

# Read the CSV files
df_ea = pd.read_csv(EA_path)
df_single = pd.read_csv(Single_path)

# --- HELPER FUNCTION: Clean Optimality ---
def clean_optimality(val):
    """
    Converts values starting with 'False' to boolean False.
    Keeps 'True' or boolean True as is.
    """
    s_val = str(val).strip()
    if s_val.lower().startswith('false'):
        return False
    elif s_val.lower().startswith('true'):
        return True
    return val

# Apply the cleaning function
df_ea['Optimality'] = df_ea['Optimality'].apply(clean_optimality)
df_single['Is Optimal'] = df_single['Is Optimal'].apply(clean_optimality)

# --- 1. Prepare EA DataFrame (File 1) ---
df_ea_clean = df_ea[[
    'Instance', 
    'Objective Value', 
    'Nodes Expanded', 
    'Total Times (s)', 
    'Optimality'
]].rename(columns={
    'Objective Value': 'EA Dual Bound Solution',
    'Nodes Expanded': 'EA Dual Bound Expanded Nodes',
    'Total Times (s)': 'EA Dual Bound Time',
    'Optimality': 'EA Dual Bound Optimality'
})

# --- 2. Prepare Single DataFrame (File 2) ---
# NOTE: We extract 'Best Known Cost' here
df_single_clean = df_single[[
    'Instance', 
    'Cost', 
    'Best Known Cost',  # <--- Getting this from the file
    'Nodes Expanded', 
    'Running Time (s)', 
    'Is Optimal'
]].rename(columns={
    'Cost': 'Single Dual Bound Solution',
    'Best Known Cost': 'Best Known Solution', # <--- Renaming it for the final output
    'Nodes Expanded': 'Single Dual Bound Expanded Nodes',
    'Running Time (s)': 'Single Dual Bound Time',
    'Is Optimal': 'Single Dual Bound Optimality'
})

# --- 3. Merge the DataFrames ---
merged_df = pd.merge(df_ea_clean, df_single_clean, on='Instance', how='outer')

# --- 4. Calculate Gaps ---
# Using the 'Best Known Solution' we extracted from the file
merged_df['EA Dual Bound Optimality Gap'] = (
    (abs(merged_df['Best Known Solution'] - merged_df['EA Dual Bound Solution']) / merged_df['Best Known Solution']) * 100
).map('{:.2f}%'.format)

merged_df['Single Dual Bound Optimality Gap'] = (
    (abs(merged_df['Best Known Solution'] - merged_df['Single Dual Bound Solution']) / merged_df['Best Known Solution']) * 100
).map('{:.2f}%'.format)

# --- 5. Format and Save ---
columns_order = [
    'Instance', 
    'Best Known Solution',
    'EA Dual Bound Solution', 'EA Dual Bound Optimality Gap', 'EA Dual Bound Expanded Nodes', 
    'EA Dual Bound Time', 'EA Dual Bound Optimality',
    'Single Dual Bound Solution', 'Single Dual Bound Optimality Gap', 'Single Dual Bound Expanded Nodes', 
    'Single Dual Bound Time', 'Single Dual Bound Optimality'
]
final_df = merged_df[columns_order]

# Save to CSV
final_df.to_csv(output_filename, index=False)

print(f"Aggregation complete. Results saved to {output_filename}")
print(final_df.head())

Aggregation complete. Results saved to aggregated_cvrptw_results.csv
      Instance  Best Known Solution  EA Dual Bound Solution  \
0     C104.txt               822.90             1402.820147   
1  C1_2_10.TXT              2643.51             4340.162547   
2     C201.txt               589.10             1090.884105   
3  C2_2_10.TXT              1806.58             4486.834387   
4     R104.txt               971.50             1451.173974   

  EA Dual Bound Optimality Gap  EA Dual Bound Expanded Nodes  \
0                       70.47%                        296076   
1                       64.18%                        110139   
2                       85.18%                        275040   
3                      148.36%                        125801   
4                       49.37%                        458703   

   EA Dual Bound Time  EA Dual Bound Optimality  Single Dual Bound Solution  \
0         1839.676529                     False                 1250.828148   
1        

In [ ]:
1

1